In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split

# --- PARAMETRI ---
K = 5         # numero di armoniche della serie di Fourier
P = 365       # periodo (es: dati giornalieri, ciclo annuale)

# Esempio: dataset your_data con colonne: Y, X1, X2, ..., Xp

df = pd.read_csv("../../data/processed/synthetic_dataset.csv")
df = df.copy()

# ------- COSTRUZIONE DELLA BASE DI FOURIER -------
t = np.arange(len(df))  # indice temporale
fourier_terms = []

for k in range(1, K + 1):
    fourier_terms.append(np.sin(2 * np.pi * k * t / P))
    fourier_terms.append(np.cos(2 * np.pi * k * t / P))

fourier_matrix = np.column_stack(fourier_terms)

# Come dataframe
fourier_df = pd.DataFrame(
    fourier_matrix,
    columns=[f"sin_{k}" if i % 2 == 0 else f"cos_{k}" 
             for k in range(1, K+1) for i in range(2)]
)

# ------- UNIONE CON LE VARIABILI X -------
X_vars = df.filter(regex="^X")      # tutte le colonne che iniziano per X
X = pd.concat([X_vars, fourier_df], axis=1)
y = df.filter(regex="^Y")

# ------- TRAIN/TEST -------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# ------- MODELLO -------
model = Lasso(alpha=1.0)
model.fit(X_train, y_train)

# ------- PREVISIONE -------
y_pred = model.predict(X_test)

print("R² test:", model.score(X_test, y_test))




R² test: 0.4901654688445737


In [6]:
#PREDICTING NEW DATA
#DOESN'T WORK YET

"""future_steps = 30
t_future = np.arange(len(df), len(df) + future_steps)

# Fourier future
fourier_future = []
for k in range(1, K + 1):
    fourier_future.append(np.sin(2 * np.pi * k * t_future / P))
    fourier_future.append(np.cos(2 * np.pi * k * t_future / P))

fourier_future = np.column_stack(fourier_future)
fourier_future_df = pd.DataFrame(
    fourier_future,
    columns=fourier_df.columns
)

# Se hai variabili X future, aggiungile qui
X_future = fourier_future_df  # in caso non ci siano X future

y_future_pred = model.predict(X_future)"""


'future_steps = 30\nt_future = np.arange(len(df), len(df) + future_steps)\n\n# Fourier future\nfourier_future = []\nfor k in range(1, K + 1):\n    fourier_future.append(np.sin(2 * np.pi * k * t_future / P))\n    fourier_future.append(np.cos(2 * np.pi * k * t_future / P))\n\nfourier_future = np.column_stack(fourier_future)\nfourier_future_df = pd.DataFrame(\n    fourier_future,\n    columns=fourier_df.columns\n)\n\n# Se hai variabili X future, aggiungile qui\nX_future = fourier_future_df  # in caso non ci siano X future\n\ny_future_pred = model.predict(X_future)'